# 19.13 Differential privacy

Differential privacy bounds how much one person can change the probability of any released result.

We audit a mechanism rather than a finished accuracy number: sensitivity sets the size of the one-row change, Laplace noise masks that change, and composition tracks repeated releases. Save a copy to Drive to edit.

In [ ]:

import math
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer, load_wine, make_blobs, make_moons
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

np.random.seed(19)


def clf_ladder():
    """D1..D5 classification ladder of rising complexity."""
    rungs = []

    x1 = np.array([[0.0, 0.0], [0.4, 0.2], [3.0, 3.0], [2.6, 3.2]])
    y1 = np.array([0, 0, 1, 1])
    rungs.append(("D1 hand 2-D points", x1, y1))

    x2, y2 = make_blobs(n_samples=200, centers=3, cluster_std=0.8, random_state=1)
    rungs.append(("D2 clean blobs (3-class)", x2, y2))

    x3, y3 = make_moons(n_samples=300, noise=0.28, random_state=2)
    rungs.append(("D3 noisy moons (non-linear)", x3, y3))

    wine = load_wine()
    rungs.append(("D4 Wine (real, 13-D, 3-class)", wine.data, wine.target))

    bc = load_breast_cancer()
    rungs.append(("D5 Breast Cancer (real, 30-D)", bc.data, bc.target))

    return rungs


def clf_accuracy(build_and_predict, X, y):
    """Split, call build_and_predict(x_tr, y_tr, x_te) -> preds, return held-out accuracy."""
    x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0, stratify=y)
    scaler = StandardScaler()
    x_tr = scaler.fit_transform(x_tr)
    x_te = scaler.transform(x_te)
    preds = build_and_predict(x_tr, y_tr, x_te)
    return accuracy_score(y_te, preds)


def fit_scaled_logistic(X, y, test_size=0.4, random_state=0):
    x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=test_size, random_state=random_state, stratify=y)
    scaler = StandardScaler()
    x_tr = scaler.fit_transform(x_tr)
    x_te = scaler.transform(x_te)
    clf = LogisticRegression(max_iter=2000)
    clf.fit(x_tr, y_tr)
    return clf, x_tr, x_te, y_tr, y_te


def fit_scaled_logistic_three_way(X, y, random_state=0):
    if len(y) < 12:
        repeats = int(np.ceil(12 / len(y)))
        X = np.tile(X, (repeats, 1))
        y = np.tile(y, repeats)
    x_tmp, x_te, y_tmp, y_te = train_test_split(X, y, test_size=0.3, random_state=random_state, stratify=y)
    x_tr, x_cal, y_tr, y_cal = train_test_split(x_tmp, y_tmp, test_size=0.35, random_state=random_state, stratify=y_tmp)
    scaler = StandardScaler()
    x_tr = scaler.fit_transform(x_tr)
    x_cal = scaler.transform(x_cal)
    x_te = scaler.transform(x_te)
    clf = LogisticRegression(max_iter=2000)
    clf.fit(x_tr, y_tr)
    return clf, x_tr, x_cal, x_te, y_tr, y_cal, y_te


def top_class_binary_view(y, proba):
    conf = proba.max(axis=1)
    pred = proba.argmax(axis=1)
    correct = (pred == y).astype(int)
    return correct, conf, pred


def degrade_d5_features(name, X):
    rng = np.random.default_rng(1918)
    if name.startswith("D5"):
        return X + rng.normal(0.0, X.std(axis=0) * 0.35, size=X.shape)
    return X


## The concept, built once

For neighboring datasets $D,D'$ and any output $o$, differential privacy asks for

$$P(M(D)=o)\le e^{\varepsilon}P(M(D')=o)+\delta.$$

On the D1 statistic, the real mechanism is a Laplace release with scale $\Delta f/\varepsilon$, and the lesson toy audit must recover total $1.600$ and share $0.625$.

In [ ]:

def dp_release(statistic, sensitivity, epsilon, delta=0.0, rng=None):
    if sensitivity <= 0:
        raise ValueError("sensitivity must be positive")
    if epsilon <= 0:
        raise ValueError("epsilon must be positive")
    if rng is None:
        rng = np.random.default_rng(1913)
    scale = sensitivity / epsilon
    noisy_value = statistic + rng.laplace(0.0, scale)
    return noisy_value, scale, delta


def dp_bound_holds(p_on_D, p_on_neighbor, epsilon, delta):
    bound = math.exp(epsilon) * p_on_neighbor + delta
    return p_on_D <= bound, bound

components = np.array([1.0, 0.5, 0.1])
total = float(components.sum())
abs_mass = float(np.abs(components).sum())
share = float(abs(components[0]) / abs_mass)
guarded = total + 0.8 * abs_mass
contrast = total - components[2]
change = contrast - total

assert np.isclose(total, 1.600)
assert np.isclose(share, 0.625)
assert np.isclose(guarded, 2.880)
assert np.isclose(change, -0.100)

ok, dp_rhs = dp_bound_holds(0.12, 0.09, epsilon=0.8, delta=1e-5)
assert ok
print("total", total, "share", share, "guarded", guarded, "bound", dp_rhs)


Now package the same release as a reusable auditing function: compute the statistic, calibrate noise to $\Delta f$, estimate utility, and use a simple membership-advantage proxy to show privacy/utility tension across the ladder.

In [ ]:

def dp_positive_rate_audit(X, y, epsilon=0.8, delta=1e-5):
    rng = np.random.default_rng(1913)
    positive = (y == np.max(y)).astype(float)
    statistic = float(positive.mean())
    sensitivity = 1.0 / len(y)
    noisy, scale, delta_used = dp_release(statistic, sensitivity, epsilon, delta, rng)
    utility_error = abs(noisy - statistic)
    privacy_risk = min(1.0, sensitivity / (scale + sensitivity))
    composition = 3.0 * epsilon
    return {
        "statistic": statistic,
        "noisy": noisy,
        "scale": scale,
        "utility_error": utility_error,
        "privacy_risk": privacy_risk,
        "composition": composition,
        "delta": delta_used,
    }


## The dataset ladder

All six notebooks in this batch use the same F15 classification ladder: a hand-checkable D1 toy, synthetic rungs, two real tabular rungs, and the real D5 Breast Cancer stress rung. The method changes, not the data-loading story.

In [ ]:

rungs = clf_ladder()

for name, X, y in rungs:
    classes, counts = np.unique(y, return_counts=True)
    preview = np.round(X[:3, : min(4, X.shape[1])], 3)
    print(name)
    print("  shape:", X.shape)
    print("  classes:", dict(zip(classes.tolist(), counts.tolist())))
    print("  sample columns:")
    print(preview)


In [ ]:

rows = []
for rung, (name, X, y) in enumerate(rungs, start=1):
    audit = dp_positive_rate_audit(X, y, epsilon=0.8)
    rows.append((rung, name, audit["privacy_risk"], audit["utility_error"], audit["composition"]))

print("rung | privacy_risk | utility_error | composed_epsilon")
for rung, name, risk, error, composed in rows:
    print(f"D{rung} | {risk:.3f} | {error:.3f} | {composed:.2f} | {name}")


In [ ]:

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
eps_grid = np.array([0.2, 0.5, 0.8, 1.2, 2.0])
summary = []

for col, (name, X, y) in enumerate(rungs):
    audit = dp_positive_rate_audit(X, y, epsilon=0.8)
    axes[0, col].bar(["raw", "DP"], [audit["statistic"], audit["noisy"]], color=["gray", "tab:blue"])
    axes[0, col].set_title(name.split("(")[0].strip(), fontsize=9)
    axes[0, col].set_ylim(0, 1)
    eps_risks = [dp_positive_rate_audit(X, y, epsilon=e)["privacy_risk"] for e in eps_grid]
    summary.append(audit["privacy_risk"])
    axes[1, col].plot(eps_grid, eps_risks, marker="o")
    axes[1, col].set_xlabel("epsilon")
    axes[1, col].set_ylim(0, 1)

fig.suptitle("DP releases and privacy-risk curves")
plt.tight_layout()
plt.figure(figsize=(6, 3))
plt.plot(range(1, 6), summary, marker="o")
plt.xlabel("ladder rung")
plt.ylabel("membership-advantage proxy")
plt.title("Metric vs. D1→D5")
plt.xticks(range(1, 6))
plt.show()


## Pitfall on D5: forgetting sensitivity

The wrong audit chooses a noise scale without $\Delta f$, so the same noise is used for four rows and hundreds of rows. The fix computes $\Delta f=1/n$ and tracks repeated epsilon spending.

In [ ]:

name, X, y = rungs[-1]
statistic = float((y == np.max(y)).mean())
wrong_scale = 1.0 / 0.8
correct_sensitivity = 1.0 / len(y)
correct_scale = correct_sensitivity / 0.8
wrong_noise_to_signal = wrong_scale / max(statistic, 1e-9)
correct_noise_to_signal = correct_scale / max(statistic, 1e-9)
composed_epsilon = 3.0 * 0.8

print("wrong scale", wrong_scale)
print("correct sensitivity", correct_sensitivity)
print("correct scale", correct_scale)
print("wrong noise/stat", wrong_noise_to_signal)
print("correct noise/stat", correct_noise_to_signal)
print("composition after three releases", composed_epsilon)


## Evaluate it + Practice

- Metric: privacy-risk or membership-advantage proxy at fixed utility.
- No-skill baseline: release the raw statistic with zero noise.
- Cheap sanity check: scale must equal sensitivity divided by epsilon.
- Ablation: set sensitivity to one for every rung and watch utility collapse.
- Failure signals: large composition budget, unbounded sensitivity, or delta treated as zero.

Practice 1: Change the random seed or stress strength and explain which rung moves most.

Practice 2: Replace positive-rate with a bounded accuracy statistic and recompute sensitivity.

Practice 3: Plot utility error against epsilon for D5 only.